# 第八阶段第二课：MLP 分类手写数字

从回归到分类：用多层感知机（MLP）识别手写数字。分类和回归的区别：输出层用 softmax，损失用交叉熵。

## 1. 准备数据：用 sklearn 自带的手写数字集

为了避免下载，这里用 sklearn 的 digits 数据集（8x8 像素的手写数字，10 类）。

In [ ]:
import torch
import torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

digits = load_digits()
X = digits.data        # 1797 张图，每张 64 个像素
y = digits.target      # 标签 0-9

X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("训练集", X_train.shape, "测试集", X_test.shape)

## 2. 定义 MLP：64 -> 32 -> 10

nn.Sequential 把层串起来。中间用 ReLU 激活函数加非线性。

In [ ]:
model = nn.Sequential(
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 10),        # 10 类输出
)
print(model)

## 3. 损失和优化器

分类任务用 CrossEntropyLoss（它内部自带 softmax）。

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=64)

## 4. 训练循环（和上一课同一套模板）

In [ ]:
for epoch in range(30):
    total_loss = 0
    for batch_x, batch_y in train_loader:
        pred = model(batch_x)
        loss = criterion(pred, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if epoch % 10 == 0:
        print(f"epoch {epoch}, 平均损失 {total_loss / len(train_loader):.4f}")

## 5. 评估准确率

In [ ]:
correct = 0
total = 0
with torch.no_grad():
    for batch_x, batch_y in test_loader:
        pred = model(batch_x)
        _, predicted = torch.max(pred, dim=1)   # 取分数最高的类别
        correct += (predicted == batch_y).sum().item()
        total += batch_y.size(0)

print(f"测试集准确率：{correct / total * 100:.1f}%")

## 6. 保存和加载模型

In [ ]:
torch.save(model.state_dict(), "mlp_digits.pt")   # 保存参数

model2 = nn.Sequential(
    nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 10),
)
model2.load_state_dict(torch.load("mlp_digits.pt"))   # 加载
print("模型已保存并重新加载")

## 7. 练习（自己动手写）

练习 1：把隐层从 32 改成 64 个神经元，重训一遍，看准确率变化。

练习 2：把训练轮数从 30 改成 50，看准确率是否更高。

In [ ]:
# 在这里写你的练习代码
